In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt

# ✅ Step 1: Load correct telecom files
all_files = glob.glob("../data/landing/*.csv")

files = [f for f in all_files if "sms-call-internet" in f]

print("Correct telecom files:", files)

# ✅ Load first 3 files
df = pd.concat([pd.read_csv(f) for f in files[:3]], ignore_index=True)

# ✅ Inspect
print("\nFirst 5 rows:")
print(df.head())

print("\nColumns:\n", df.columns)

# ✅ Step 2: FIX COLUMN NAMES (IMPORTANT ✅)
df = df.rename(columns={
    'datetime': 'timestamp',
    'CellID': 'grid_id',
    'internet': 'internet_usage'
})

# ✅ Create required columns (IMPORTANT ✅)
df['call_count'] = df['callin'] + df['callout']
df['sms_count'] = df['smsin'] + df['smsout']

# ✅ Step 3: Data quality checks
print("\nMissing timestamps:", df['timestamp'].isnull().sum())

invalid_rows = df[
    (df['call_count'] < 0) |
    (df['sms_count'] < 0) |
    (df['internet_usage'] < 0)
]
print("Invalid rows:", len(invalid_rows))

print("Duplicate rows:", df.duplicated().sum())

# ✅ Step 4: Convert timestamp (already datetime format)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

df = df.dropna(subset=['timestamp'])

# ✅ Extract features
df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.date

# ✅ Step 5: Stats
print("\nData Types:\n", df.dtypes)
print("\nSummary:\n", df.describe())
print("\nMissing Values:\n", df.isnull().sum())

# ✅ Step 6: Chart 1 — Daily Usage
daily_usage = df.groupby('day')['internet_usage'].sum()

daily_usage.plot(figsize=(10,5), title="Daily Internet Usage")
plt.xlabel("Date")
plt.ylabel("Internet Usage")
plt.show()

# ✅ Step 7: Chart 2 — Top Regions
region_usage = df.groupby('grid_id')['internet_usage'].sum().sort_values(ascending=False).head(10)

region_usage.plot(kind='bar', title="Top 10 Regions by Usage")
plt.xlabel("Grid ID")
plt.ylabel("Usage")
plt.show()

Correct telecom files: ['../data/landing\\sms-call-internet-mi-2013-11-01.csv', '../data/landing\\sms-call-internet-mi-2013-11-02.csv', '../data/landing\\sms-call-internet-mi-2013-11-03.csv', '../data/landing\\sms-call-internet-mi-2013-11-04.csv', '../data/landing\\sms-call-internet-mi-2013-11-05.csv', '../data/landing\\sms-call-internet-mi-2013-11-06.csv', '../data/landing\\sms-call-internet-mi-2013-11-07.csv']

First 5 rows:
              datetime  CellID  countrycode   smsin  smsout  callin  callout  \
0  2013-11-01 00:00:00       1            0  0.3521     NaN     NaN   0.0273   
1  2013-11-01 00:00:00       1           33     NaN     NaN     NaN      NaN   
2  2013-11-01 00:00:00       1           39  1.7322  1.1047  0.5919   0.4020   
3  2013-11-01 00:00:00       2            0  0.3581     NaN     NaN   0.0273   
4  2013-11-01 00:00:00       2           33     NaN     NaN     NaN      NaN   

   internet  
0       NaN  
1    0.0261  
2   57.7729  
3       NaN  
4    0.0274  

Col

KeyError: 'call_in'